# ST 554 Homework 9
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

In [1]:
#importing required modules
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

## Reading in Data
For this assignment, I chose to use a dataset on Florida real estate properties sold in 2026. I downloaded this dataset from Kaggle, and it is linked [here](https://www.kaggle.com/datasets/kanchana1990/florida-real-estate-sold-dataset-2026). The data file will also be available in my Github, titled `florida_real_estate_sold_properties_utlimate.csv`.

### Creating a Spark Session
The code below creates a spark session for use with reading in our data and building our models.

In [2]:
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 22:46:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


The code below only allows error messages to print out going forward.

In [3]:
spark.sparkContext.setLogLevel("ERROR")

### Importing Data
The code below reads in our data using `pandas`. The `.dropna()` method is also used to drop rows where `NaN` values are present within certain variables.

In [4]:
house_data = pd.read_csv("florida_real_estate_sold_properties_ultimate.csv") \
             .dropna(subset=['sqft', 'year_built'])
house_data.head()

,type,sub_type,listPrice,lastSoldPrice,sqft,stories,beds,baths,baths_full,baths_full_calc,garage,year_built,zip,sanitized_text
0,single_family,NaN,630000.0,605000,2274.0,1.0,2.0,3.0,2.0,2.0,2.0,2007.0,33446.0,"Beautiful 2 Bedroom + Den, 2.5 Bath Home - Mov..."
1,single_family,NaN,289000.0,285000,2170.0,1.0,3.0,2.0,2.0,2.0,2.0,1980.0,33876.0,Welcome to Florida living at its best! This 3-...
2,condos,condo,449000.0,425000,1722.0,NaN,3.0,2.0,2.0,2.0,2.0,2016.0,33913.0,Best Value in Casella and priced to sell... St...
3,single_family,NaN,599000.0,596000,1699.0,1.0,3.0,3.0,3.0,3.0,NaN,1952.0,33009.0,"Beautifully renovated 3-bedroom, 3-bathroom ho..."
4,single_family,NaN,173500.0,165000,640.0,1.0,1.0,1.0,1.0,1.0,NaN,1971.0,32118.0,Experience the ultimate beachfront lifestyle i...


Now, we convert our `pandas dataframe` to a `spark` SQL style dataframe.

In [6]:
FL_houses = spark.createDataFrame(house_data)
FL_houses.show(5)

+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+
|         type|sub_type|listPrice|lastSoldPrice|  sqft|stories|beds|baths|baths_full|baths_full_calc|garage|year_built|    zip|      sanitized_text|
+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+
|single_family|     NaN| 630000.0|       605000|2274.0|    1.0| 2.0|  3.0|       2.0|            2.0|   2.0|    2007.0|33446.0|Beautiful 2 Bedro...|
|single_family|     NaN| 289000.0|       285000|2170.0|    1.0| 3.0|  2.0|       2.0|            2.0|   2.0|    1980.0|33876.0|Welcome to Florid...|
|       condos|   condo| 449000.0|       425000|1722.0|    NaN| 3.0|  2.0|       2.0|            2.0|   2.0|    2016.0|33913.0|Best Value in Cas...|
|single_family|     NaN| 599000.0|       596000|1699.0|    1.0| 3.0|  3.0|       3.0|            3.0|   Na

## Splitting the Data, Metrics, and Models
This section involves:
- splitting the data into a training and test set using spark MLlib
- choosing and describing a metric for judging the fitted models
- fitting three different classes of models and describing them

### Splitting the Data
The code below uses spark MLlib to split the data into a training and test set. This is done by using the `.randomSplit()` method on a `spark` SQL style dataframe.

In [8]:
train, test = FL_houses.randomSplit([0.8,0.2], seed = 44)
print('Number of training observations:', train.count())
print('Number of test observations:', test.count())

Number of training observations: 8038
Number of test observations: 2070


### Choosing a Metric
For this assignment, I will choose the root mean square error (RMSE) for judging the models. RMSE is a very common metric and is easy to interpret. The lower the RMSE, the better the model fit. The mean square error (MSE) is the average of the difference between observed values and values predicted by the model. We want to minimze MSE because we want our model predictions to be close to what is observed! As such, the RMSE is simply the square-root of the MSE.

### Classes of Models
Since I am interested in using `lastSoldPrice` as the response, I will fit three different classes of *regression* models. The following models will be fit:
- A **linear regression model**
- A **decision tree regression model**
- A **random forest regression model**

## Model Fitting
In this section, we use Spark MLlib to fit the three different classes of models described previously to the training data. A pipeline in `pyspark` will be set up for each model. Cross validation will be used to choose the best model for each model type. These best models will be compared using RMSE; but remember, the test data will be used to select the best model overall!

### Model 1: Linear Regression Model
The first model we will fit is a linear regression model that uses the following features to predict `lastSoldPrice`:
- `sqft`
- a multi-story indicator variable that will be calculated using the `stories` variable
    - will be called `multi_story_ind`
- `year_built`

**Note:** This is the model that will feature 4 transformations in the pipeline!

We first need to create our multi-story indicator variable. This can be done using `Binarizer()`. We will use a threshold of 1.5 since anything lower would clearly indicate that the home is not multi-story, whereas anything larger would clearly indicate that the home is multi-story.

In [9]:
binaryTrans1 = Binarizer(threshold = 1.5, inputCol = "stories", outputCol = "multi_story_ind")
binaryTrans1.transform(FL_houses).show(5)

+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+---------------+
|         type|sub_type|listPrice|lastSoldPrice|  sqft|stories|beds|baths|baths_full|baths_full_calc|garage|year_built|    zip|      sanitized_text|multi_story_ind|
+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+---------------+
|single_family|     NaN| 630000.0|       605000|2274.0|    1.0| 2.0|  3.0|       2.0|            2.0|   2.0|    2007.0|33446.0|Beautiful 2 Bedro...|            0.0|
|single_family|     NaN| 289000.0|       285000|2170.0|    1.0| 3.0|  2.0|       2.0|            2.0|   2.0|    1980.0|33876.0|Welcome to Florid...|            0.0|
|       condos|   condo| 449000.0|       425000|1722.0|    NaN| 3.0|  2.0|       2.0|            2.0|   2.0|    2016.0|33913.0|Best Value in Cas...|            0.0|
|single_fa

Next, we use the `SQLTransformer()` to select the columns that we want and rename the response (`lastSoldPrice`) as `label`. We will use an additional "transformation" here to take the log of the `sqft` variable as well.

In [10]:
sqlTrans1 = SQLTransformer(
    statement = """
                SELECT log(sqft) as log_sqft, multi_story_ind, year_built, lastSoldPrice as label FROM __THIS__
                """
    )

sqlTrans1.transform(
    binaryTrans1.transform(FL_houses)
).show(5)

+------------------+---------------+----------+------+
|          log_sqft|multi_story_ind|year_built| label|
+------------------+---------------+----------+------+
| 7.729295674310482|            0.0|    2007.0|605000|
| 7.682482446534506|            0.0|    1980.0|285000|
| 7.451241684987676|            0.0|    2016.0|425000|
|7.4377951216719325|            0.0|    1952.0|596000|
| 6.461468176353717|            0.0|    1971.0|165000|
+------------------+---------------+----------+------+
only showing top 5 rows


Continuing on, we need to put our model features into a single column called `features`. We can do this with `VectorAssembler()`.

In [11]:
assembler1 = VectorAssembler(
                inputCols = ["log_sqft", "multi_story_ind", "year_built"],
                outputCol = "features",
                handleInvalid = 'keep'
            )

assembler1.transform(
    sqlTrans1.transform(
        binaryTrans1.transform(FL_houses)
    )
).show(5)

+------------------+---------------+----------+------+--------------------+
|          log_sqft|multi_story_ind|year_built| label|            features|
+------------------+---------------+----------+------+--------------------+
| 7.729295674310482|            0.0|    2007.0|605000|[7.72929567431048...|
| 7.682482446534506|            0.0|    1980.0|285000|[7.68248244653450...|
| 7.451241684987676|            0.0|    2016.0|425000|[7.45124168498767...|
|7.4377951216719325|            0.0|    1952.0|596000|[7.43779512167193...|
| 6.461468176353717|            0.0|    1971.0|165000|[6.46146817635371...|
+------------------+---------------+----------+------+--------------------+
only showing top 5 rows


We are now ready to set up a pipeline for this model and fit a multiple linear regression (MLR) model. A few important notes:
- The `Pipeline()` function from `pyspark.ml` will be used to set up our sequence of transformations and estimators
- Since `LinearRegression()` does regularized regression, we will provide a list of values for both `regParam` and `elasticNetParam` so that cross-validation can be used to select the best model!

In [13]:
# set up instance of our linear regression
lr = LinearRegression()

# build parameter grid
paramGrid1 = ParamGridBuilder() \
             .addGrid(lr.regParam, [0, 0.25, 0.5, 0.75, 0.1]) \
             .addGrid(lr.elasticNetParam, [0, 0.5, 0.75, 0.9, 1]) \
             .build()

# create pipeline
pipeline1 = Pipeline(stages = [binaryTrans1, sqlTrans1, assembler1, lr])

# set up cross validation with pipeline
crossval1 = CrossValidator(estimator = pipeline1,
                           estimatorParamMaps = paramGrid1,
                           evaluator = RegressionEvaluator(metricName = 'rmse'),
                           numFolds = 5,
                           seed = 44)

# fit the model using cross-validation to choose the best model
cvModel1 = crossval1.fit(train)

Although the `.bestModel` attribute will store the best model as a result of cross-validation by default, we want to look to see what model is the best. The code below prints the RMSE for the model fit with its corresponding parameters, `regParam` and `elasticNetParam`. I then sort this output in ascending order so we can see the lowest RMSE first (along with its corresponding parameters.

In [14]:
my_list1 = []

for i in range(len(paramGrid1)):
    my_list1.append([cvModel1.avgMetrics[i], paramGrid1[i].values()])
    
my_list1.sort()
my_list1

[[1013681.736796213, dict_values([0.75, 0.5])],
 [1013681.7433536643, dict_values([0.5, 0.75])],
 [1013681.7445546264, dict_values([0.75, 1.0])],
 [1013681.7487362707, dict_values([0.75, 0.9])],
 [1013681.7531767624, dict_values([0.5, 0.5])],
 [1013681.7550088011, dict_values([0.75, 0.75])],
 [1013681.7597342425, dict_values([0.25, 1.0])],
 [1013681.7611281082, dict_values([0.25, 0.9])],
 [1013681.7632188702, dict_values([0.25, 0.75])],
 [1013681.7650509455, dict_values([0.5, 1.0])],
 [1013681.7667034948, dict_values([0.25, 0.5])],
 [1013681.7678387619, dict_values([0.5, 0.9])],
 [1013681.7720318608, dict_values([0.1, 1.0])],
 [1013681.7799087244, dict_values([0.75, 0.0])],
 [1013681.7864662539, dict_values([0.5, 0.0])],
 [1013681.7930238347, dict_values([0.25, 0.0])],
 [1013681.7938427369, dict_values([0.1, 0.75])],
 [1013681.7940209846, dict_values([0.1, 0.9])],
 [1013681.7952365351, dict_values([0.1, 0.5])],
 [1013681.7969583876, dict_values([0.1, 0.0])],
 [1013681.7995814101, dict_

Thus, the best multiple linear regression model is one with a regularization parameter of 0.75 and an elastic net parameter of 0.5. We can now print the intercept and coefficients of this best model.

In [15]:
# need to use indexing since the model is the last stage of the pipeline
print('Intercept:', cvModel1.bestModel.stages[-1].intercept)
print('log_sqft, multi_story_ind, year_built coeffs:', cvModel1.bestModel.stages[-1].coefficients)
print('RMSE:', round(my_list1[0][0], 5))

Intercept: 3109038.5506007755
log_sqft, multi_story_ind, year_built coeffs: [1271431.3117593408,71740.09757702916,-6003.523274794319]
RMSE: 1013681.7368
